## Requirements

### Java
PySpark requires Java to run. **Java 17 is required** — newer versions (18+) are not fully compatible with the current Spark version.

1. Download and install **Eclipse Temurin JDK 17** from:  
   https://adoptium.net/temurin/releases/?version=17

2. After installing, set `JAVA_HOME` in Windows System Environment Variables
   to the JDK 17 installation path, e.g.:  
   `C:\Users\YourUser\AppData\Local\Programs\Eclipse Adoptium\jdk-17.x.x-hotspot`

3. Confirm the correct version is active:
```bash
   java -version
   # Expected: openjdk version "17.x.x"
```

### Python
- Python 3.8 or higher

### Libraries
Install the required library via pip:

```bash
pip install pyspark
```

In [61]:
import pandas as pd
from pyspark.sql.types import *
from pyspark.sql import SparkSession
from collections import Counter
from pyspark.sql import functions as F
import glob

In [62]:
spark = SparkSession.builder \
    .appName("AnalisePatricia") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(spark.sparkContext._jvm.System.getProperty("java.version"))

17.0.19


In [63]:
files = glob.glob("data/2020/*.parquet")
print(files)

['data/2020\\yellow_tripdata_2020-01.parquet', 'data/2020\\yellow_tripdata_2020-02.parquet', 'data/2020\\yellow_tripdata_2020-03.parquet', 'data/2020\\yellow_tripdata_2020-04.parquet', 'data/2020\\yellow_tripdata_2020-05.parquet', 'data/2020\\yellow_tripdata_2020-06.parquet', 'data/2020\\yellow_tripdata_2020-07.parquet', 'data/2020\\yellow_tripdata_2020-08.parquet', 'data/2020\\yellow_tripdata_2020-09.parquet', 'data/2020\\yellow_tripdata_2020-10.parquet', 'data/2020\\yellow_tripdata_2020-11.parquet', 'data/2020\\yellow_tripdata_2020-12.parquet']


In [65]:
schema = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampNTZType(), True),
    StructField("tpep_dropoff_datetime", TimestampNTZType(), True),
    StructField("passenger_count", DoubleType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", DoubleType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("airport_fee", DoubleType(), True),  # forçamos DOUBLE em todos
])

df = spark.read.schema(schema).parquet(*files)
df.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [66]:
df.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|1       |2020-01-01 00:28:15 |2020-01-01 00:33:03  |1.0            |1.2          |1.0       |N                 |238         |239         |1           |6.0        |3.0  |0.5    |1.47     

In [68]:
for f in files:
    temp_df = spark.read.parquet(f)
    print(f"{f.split('/')[-1]} → {temp_df.count()} linhas, {len(temp_df.columns)} colunas")

2020\yellow_tripdata_2020-01.parquet → 6405008 linhas, 19 colunas
2020\yellow_tripdata_2020-02.parquet → 6299367 linhas, 19 colunas
2020\yellow_tripdata_2020-03.parquet → 3007687 linhas, 19 colunas
2020\yellow_tripdata_2020-04.parquet → 238073 linhas, 19 colunas
2020\yellow_tripdata_2020-05.parquet → 348415 linhas, 19 colunas
2020\yellow_tripdata_2020-06.parquet → 549797 linhas, 19 colunas
2020\yellow_tripdata_2020-07.parquet → 800412 linhas, 19 colunas
2020\yellow_tripdata_2020-08.parquet → 1007286 linhas, 19 colunas
2020\yellow_tripdata_2020-09.parquet → 1341017 linhas, 19 colunas
2020\yellow_tripdata_2020-10.parquet → 1681132 linhas, 19 colunas
2020\yellow_tripdata_2020-11.parquet → 1509000 linhas, 19 colunas
2020\yellow_tripdata_2020-12.parquet → 1461898 linhas, 19 colunas


### **Dataset Attributes**

| Field Name | Type | Description |
|---|---|---|
| `VendorID` | Long | Identifies the technology provider that recorded the trip. **1** = Creative Mobile Technologies, **2** = VeriFone Inc. |
| `tpep_pickup_datetime` | Timestamp | Date and time when the taximeter was engaged (trip start). |
| `tpep_dropoff_datetime` | Timestamp | Date and time when the taximeter was disengaged (trip end). |
| `passenger_count` | Double | Number of passengers in the vehicle, manually entered by the driver. |
| `trip_distance` | Double | Total trip distance in miles as reported by the taximeter. |
| `RatecodeID` | Double | Rate code applied at the end of the trip. **1** = Standard, **2** = JFK, **3** = Newark, **4** = Nassau/Westchester, **5** = Negotiated, **6** = Group ride. |
| `store_and_fwd_flag` | String | Indicates whether the trip record was temporarily stored in the vehicle due to lack of server connection before being transmitted. **Y** = stored, **N** = not stored. |
| `PULocationID` | Long | TLC Taxi Zone ID where the passenger was picked up. |
| `DOLocationID` | Long | TLC Taxi Zone ID where the passenger was dropped off. |
| `payment_type` | Long | Payment method used. **1** = Credit card, **2** = Cash, **3** = No charge, **4** = Dispute, **5** = Unknown, **6** = Voided. |
| `fare_amount` | Double | Base fare calculated by the taximeter based on time and distance. |
| `extra` | Double | Miscellaneous surcharges, including **$0.50** rush hour and **$1.00** overnight supplements. |
| `mta_tax` | Double | Fixed **$0.50** tax automatically applied to all standard-rate trips, funding the Metropolitan Transportation Authority (NYC public transit). |
| `improvement_surcharge` | Double | Fixed **$0.30** surcharge levied at the start of every trip since 2015, funding wheelchair-accessible taxi improvements. |
| `tip_amount` | Double | Tip amount, automatically captured for credit card payments only. Cash tips are not recorded. |
| `tolls_amount` | Double | Total amount of road tolls paid during the trip (bridges, tunnels, etc.). |
| `total_amount` | Double | Total amount charged to the passenger, including all fees and surcharges. Does not include cash tips. |
| `congestion_surcharge` | Double | Surcharge of **$2.50** applied to trips entering or leaving Manhattan below 96th Street, introduced in 2019 to fund public transport improvements. Not applicable to all trips, hence the missing values. |
| `airport_fee` | Double | Fixed **$1.25** surcharge for pickups at LaGuardia (LGA) and John F. Kennedy (JFK) airports. **Not applicable in 2020** — introduced by NYC TLC in 2022. |

### **Dimensions**

In [69]:
# ── 1. DIMENSÕES GERAIS ──────────────────────────────────────────
total_rows = df.count()
total_cols = len(df.columns)
print(f"Total de linhas: {total_rows:,}")
print(f"Total de colunas: {total_cols}")

Total de linhas: 24,649,092
Total de colunas: 19


### **Data Types**

In [70]:
# ── 2. SCHEMA POR TIPOS ─────────────────────────────────────────
type_counts = Counter([str(f.dataType) for f in df.schema.fields])
for t, count in type_counts.items():
    print(f"{t}: {count} colunas")

LongType(): 4 colunas
TimestampNTZType(): 2 colunas
DoubleType(): 12 colunas
StringType(): 1 colunas


The dataset comprises **19 attributes** distributed across 4 data types:

- **LongType** (4 columns): Integer fields representing IDs and categorical codes — `VendorID`, `PULocationID`, `DOLocationID`, and `payment_type`.

- **TimestampNTZType** (2 columns): Datetime fields without timezone — `tpep_pickup_datetime` and `tpep_dropoff_datetime`.

- **DoubleType** (12 columns): Continuous numerical fields representing distances, fare components, and trip metadata.

- **StringType** (1 column): The `store_and_fwd_flag` field, indicating whether the trip record was temporarily stored before transmission (Y/N).

### **Temporal Range Validation**

In [71]:
# ── 3. INTERVALO TEMPORAL ───────────────────────────────────────
df.select(
    F.min("tpep_pickup_datetime").alias("data_inicio"),
    F.max("tpep_pickup_datetime").alias("data_fim")
).show()

+-------------------+-------------------+
|        data_inicio|           data_fim|
+-------------------+-------------------+
|2002-12-31 23:06:55|2021-06-10 10:10:48|
+-------------------+-------------------+



In [72]:
invalid_dates = df.filter(
    (F.col("tpep_pickup_datetime") < "2020-01-01") |
    (F.col("tpep_pickup_datetime") > "2020-12-31")
).count()

print(f"Linhas com datas fora de 2020: {invalid_dates:,}")
print(f"Percentagem: {invalid_dates / total_rows * 100:.4f}%")

Linhas com datas fora de 2020: 45,072
Percentagem: 0.1829%


In [73]:
df = df.filter(
    (F.col("tpep_pickup_datetime") >= "2020-01-01") &
    (F.col("tpep_pickup_datetime") <= "2020-12-31")
)

In [74]:
df.select(
    F.min("tpep_pickup_datetime").alias("data_inicio"),
    F.max("tpep_pickup_datetime").alias("data_fim")
).show()

+-------------------+-------------------+
|        data_inicio|           data_fim|
+-------------------+-------------------+
|2020-01-01 00:00:00|2020-12-31 00:00:00|
+-------------------+-------------------+



The dataset was filtered to retain only records within the year 2020. A total of **45,072 records** (0.18% of the dataset) presented `tpep_pickup_datetime` values outside this range — either predating 2020 or extending beyond December 31, 2020 — and were removed as 
invalid entries.

### **Missing Values**

In [75]:
total_rows = df.count()

# Calcula tudo em PySpark sem collect()
missing_table = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns
]).unpivot([], df.columns, "column", "missing_count") \
  .withColumn("missing_pct", F.round(F.col("missing_count") / total_rows * 100, 4)) \
  .orderBy("missing_count", ascending=False)

missing_table.show(truncate=False)

+---------------------+-------------+-----------+
|column               |missing_count|missing_pct|
+---------------------+-------------+-----------+
|airport_fee          |24603999     |99.9999    |
|passenger_count      |806563       |3.2782     |
|RatecodeID           |806563       |3.2782     |
|store_and_fwd_flag   |806563       |3.2782     |
|congestion_surcharge |806563       |3.2782     |
|VendorID             |0            |0.0        |
|tpep_pickup_datetime |0            |0.0        |
|tpep_dropoff_datetime|0            |0.0        |
|trip_distance        |0            |0.0        |
|PULocationID         |0            |0.0        |
|DOLocationID         |0            |0.0        |
|payment_type         |0            |0.0        |
|fare_amount          |0            |0.0        |
|extra                |0            |0.0        |
|mta_tax              |0            |0.0        |
|tip_amount           |0            |0.0        |
|tolls_amount         |0            |0.0        |


Profiling revealed missing values in **5 of the 19 attributes**. Four fields — `passenger_count`, `RatecodeID`, `store_and_fwd_flag`, and `congestion_surcharge` — each present **806,563 null records** (~3.28% of the dataset), likely due to occasional failures in the taxi meter recording system.

The `airport_fee` field shows a near-total absence of values (**~99.99%**), which is expected: this surcharge was only introduced by the NYC TLC in 2022 and did not exist in 2020. This column will be dropped prior to analysis.

All remaining **14 attributes** contain no missing values.

In [76]:
# Dropar airport_fee (não existia em 2020)
df = df.drop("airport_fee")

In [77]:
# Preencher os nulos das restantes 4 colunas
df = df.fillna({
    "passenger_count": 0,
    "RatecodeID": 1,          # 1 = Standard rate (valor default NYC TLC)
    "store_and_fwd_flag": "N", # N = não foi armazenado
    "congestion_surcharge": 0
})

In [78]:
# ── 5. LINHAS COM PELO MENOS UM NULO ────────────────────────────
has_null = df.filter(
    F.greatest(*[F.col(c).isNull().cast("int") for c in df.columns]) == 1
).count()
print(f"Linhas com pelo menos 1 nulo: {has_null:,} ({has_null/total_rows*100:.1f}%)")

Linhas com pelo menos 1 nulo: 0 (0.0%)


### **Duplicate Records**

In [79]:
# ── 6. DUPLICADOS ───────────────────────────────────────────────
duplicates = total_rows - df.distinct().count()
print(f"Duplicados: {duplicates:,}")

Duplicados: 12,950


In [80]:
# Antes
total_before = df.count()

# Remover duplicados
df = df.distinct()

# Depois
total_after = df.count()

print(f"Linhas antes: {total_before:,}")
print(f"Linhas depois: {total_after:,}")
print(f"Duplicados removidos: {total_before - total_after:,}")

Linhas antes: 24,604,020
Linhas depois: 24,591,070
Duplicados removidos: 12,950


A total of **12,950 duplicate records** were identified and removed using `distinct()`. These exact-match duplicates are likely the result of data ingestion errors in the NYC TLC reporting system. After removal, the dataset retains **24,591,070 records**.

### **Feature Engineering**

In [81]:
# ── 7. FEATURE ENGINEERING ──────────────────────────────────────
df = df.withColumn("trip_duration",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60) \
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime")) \
    .withColumn("pickup_day_of_week", F.dayofweek("tpep_pickup_datetime")) \
    .withColumn("pickup_month", F.month("tpep_pickup_datetime")) \
    .withColumn("is_rush_hour",
        ((F.hour("tpep_pickup_datetime").between(7, 9)) |
         (F.hour("tpep_pickup_datetime").between(16, 19))).cast("boolean")) \
    .withColumn("is_weekend",
        F.dayofweek("tpep_pickup_datetime").isin([1, 7]).cast("boolean")) \
    .withColumn("fare_per_mile",
        F.when(F.col("trip_distance") > 0, F.col("fare_amount") / F.col("trip_distance")))

print(f"Colunas após feature engineering: {len(df.columns)}")
df.printSchema()

Colunas após feature engineering: 25
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = false)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = false)
 |-- store_and_fwd_flag: string (nullable = false)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = false)
 |-- trip_duration: double (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- pickup_day_of_week: integer (nullable =

To support later querying and analysis tasks, **7 new features** were engineered from existing fields using `withColumn()` and Spark's built-in datetime functions.

- **`trip_duration`**: Duration of the trip in minutes, computed as the difference between `tpep_dropoff_datetime` and `tpep_pickup_datetime`.

- **`pickup_hour`**: Hour of the day extracted from `tpep_pickup_datetime`.

- **`pickup_day_of_week`**: Day of the week extracted from `tpep_pickup_datetime` (1 = Sunday, 7 = Saturday).

- **`pickup_month`**: Month extracted from `tpep_pickup_datetime`.

- **`is_rush_hour`**: Boolean indicator of whether the trip occurred during NYC peak traffic hours (7:00–9:00 and 16:00–19:00).

- **`is_weekend`**: Boolean indicator of whether the trip took place on a Saturday or Sunday.

- **`fare_per_mile`**: Fare amount divided by trip distance in miles, representing the cost efficiency of the trip. Only computed for trips with `trip_distance > 0`.

After feature engineering, the final dataset contains **25 attributes**.